# The one FMP pull, before cancellation

**Carlo Hofer.** Run this in Google Colab. It downloads everything the research programme
needs from Financial Modeling Prep, writes a manifest, and runs a set of checks whose
numbers decide whether the subscription can safely be cancelled.

## Read this first

After cancellation nothing here can be downloaded again. So the job is not "run the code"
— it is **prove you got everything, before you cancel**.

FMP has confirmed in writing that retained data may be kept: personal use, on a local
computer, never published. Note the second condition. **A Google Drive copy is not a local
copy.** This notebook writes to Drive because Colab disconnects; the last section tells you
how to bring it down, verify it, and clear Drive.

## The order things run in, and why

| block | what | why this position |
|---|---|---|
| 0 | tiers 1 and 2 for the existing 304/302 | already planned, unfinished |
| 1a | S&P 500 current list + constituent history | everything in block 1 depends on the union it builds |
| 2 | indices, ETFs, treasury rates | small; run early so a later timeout cannot lose it |
| 1c | prices, dividends, splits, market cap, profile for the union | the bulk of the work |
| 1d | delisted-companies, full pagination | matches removed tickers to delisting dates |
| 3 | fundamentals for the union | first to drop if time is short |
| 4 | Euronext Amsterdam | optional, only if everything above is done |
| 0t3 | tier 3 for the 302 | last, if at all — Paper 1's event study is shelved |

Work top to bottom. **Do not use Runtime → Run all.** Sections 2 and 3 are gates: if a gate
fails, stop and read it.

---
## 1. Setup

Nothing to install. This mounts Drive so the download survives a disconnect. Google will
ask permission; click through it.

In [ ]:
import os, json, gzip, time, random, hashlib, re
from datetime import datetime, timezone, date
from pathlib import Path
from collections import defaultdict
import requests

from google.colab import drive, userdata
drive.mount('/content/drive')

OUT        = Path('/content/drive/MyDrive/fmp_extraction_2026-09')
RAW        = OUT / 'raw'
VENDOR_DCF = OUT / 'vendor_dcf_DO_NOT_OPEN'
LEDGER     = OUT / 'manifest_entries.jsonl'

for d in (OUT, RAW, VENDOR_DCF):
    d.mkdir(parents=True, exist_ok=True)

guard = VENDOR_DCF / 'DO_NOT_OPEN.md'
if not guard.exists():
    guard.write_text(
        "Do not open these files.\n\n"
        "FMP's own DCF valuations. A check on your valuation model, to be compared only\n"
        "AFTER your model is finished and its assumptions written down and dated.\n")

print('Saving everything to:', OUT)

### Your API key

The key goes in **Colab's Secrets panel**, never in a cell. Anything typed into a cell is
saved inside the notebook file and can reach GitHub.

Key icon (🔑) in the left sidebar → **Add new secret** → name it exactly `FMP_API_KEY` →
paste the key → switch on **Notebook access**.

In [ ]:
try:
    API_KEY = userdata.get('FMP_API_KEY')
except Exception:
    API_KEY = None

assert API_KEY, (
    "No FMP_API_KEY found.\n"
    "Key icon in the left sidebar, add a secret named exactly FMP_API_KEY, paste the key,\n"
    "switch on Notebook access, then run this cell again.")

print('Key loaded, length', len(API_KEY), '- the key itself is never printed.')

---
## 2. GATE: what does this key actually serve?

On 2 September the FMP connection used for testing was on a restricted key: it served only
a handful of demo symbols and refused the index, directory and statement groups entirely.
On 7 September the same connection answered all of them, so the key in use had changed.

Either way this gate is what proves it for *your* key, on the day, before 30,000 requests.

In [ ]:
BASE = 'https://financialmodelingprep.com/stable'

def try_once(path, **params):
    """One request. Returns (status_code, parsed json or None)."""
    params['apikey'] = API_KEY
    try:
        r = requests.get(f'{BASE}/{path}', params=params, timeout=30)
    except Exception:
        return 0, None
    if r.status_code != 200:
        return r.status_code, None
    try:
        return 200, r.json()
    except Exception:
        return 200, None

GATE = [
    ('daily prices, US',      'historical-price-eod/full',        dict(symbol='DUK'),      'BLOCKING'),
    ('daily prices, Europe',  'historical-price-eod/full',        dict(symbol='ENGI.PA'),  'BLOCKING'),
    ('dividend-adjusted',     'historical-price-eod/dividend-adjusted', dict(symbol='DUK'), 'BLOCKING'),
    ('historical market cap', 'historical-market-capitalization',  dict(symbol='DUK'),     'BLOCKING'),
    ('company profile',       'profile',                           dict(symbol='DUK'),     'BLOCKING'),
    ('delisted companies',    'delisted-companies',                dict(),                 'BLOCKING'),
    ('S&P 500 current',       'sp500-constituent',                 dict(),                 'BLOCKING'),
    ('S&P 500 history',       'historical-sp500-constituent',      dict(),                 'BLOCKING'),
    ('index series',          'historical-price-eod/full',         dict(symbol='^GSPC'),   'BLOCKING'),
    ('total-return index',    'historical-price-eod/full',         dict(symbol='^SP500TR'), 'needed for check 6'),
    ('treasury rates',        'treasury-rates',                    dict(),                 'needed'),
    ('income statement',      'income-statement',                  dict(symbol='DUK', period='annual', limit=2), 'needed'),
    ('exchange list',         'available-exchanges',               dict(),                 'needed for block 4'),
    ('earnings transcripts',  'earning-call-transcript-dates',     dict(symbol='DUK'),     'tier 3, lowest'),
    ('ESG scores',            'esg-ratings',                       dict(symbol='DUK'),     'tier 3, lowest'),
]

print(f"{'what':<24}{'result':<14}{'importance'}")
print('-' * 64)
blocked, denied_any = [], []
for label, path, params, importance in GATE:
    status, data = try_once(path, **params)
    ok = status == 200 and data not in (None, [], {})
    verdict = 'OK' if ok else (f'DENIED {status}' if status else 'no answer')
    if not ok:
        denied_any.append((label, status))
        if importance == 'BLOCKING':
            blocked.append(label)
    print(f'{label:<24}{verdict:<14}{importance}')
    time.sleep(0.4)

print()
if blocked:
    print('STOP. These are blocking and your key refused them:')
    for b in blocked:
        print('   -', b)
    print('\nCheck your plan at financialmodelingprep.com while logged in.')
    print('Do not continue: cancelling after a partial pull makes the gap permanent.')
else:
    print('All blocking endpoints answered.')
    if denied_any:
        print('\nRefused, non-blocking - record these in the handover as things you never had:')
        for label, status in denied_any:
            print(f'   - {label} (HTTP {status})')

### The default-window trap

**If you do not give FMP a date range, it returns roughly the last five years and says
nothing.** No error, no warning. It looks exactly like a complete answer.

Verified 2 September: asked without dates, `AAPL` began 2021-09-03; asked
`from=2010-01-04`, the same endpoint returned 2010 rows happily.

Every dated request below passes an explicit `from` and `to`. This confirms your key
honours them.

In [ ]:
status, data = try_once('historical-price-eod/full', symbol='AAPL',
                        **{'from': '1990-01-01', 'to': '2026-09-07'})
if status != 200 or not data:
    print('Could not check - endpoint returned', status)
else:
    rows = data if isinstance(data, list) else data.get('historical', [])
    dates = sorted(r['date'] for r in rows if isinstance(r, dict) and 'date' in r)
    print('rows:', len(rows), ' earliest:', dates[0] if dates else 'none',
          ' latest:', dates[-1] if dates else 'none')
    print()
    if dates and dates[0] > '2011-01-01':
        print('WARNING: asked for 1990 onward, earliest row is', dates[0] + '.')
        print('The date range is being ignored. Do not run the download.')
    else:
        print('Date range honoured. Continue.')

---
## 3. The symbol lists

Four universes, each built and saved before anything is downloaded.

In [ ]:
def sanitise(sym):
    """Filename-safe symbol. Keeps ^ and . which FMP uses; replaces path characters."""
    return re.sub(r'[/\\\\:*?"<>|]', '_', sym)

def save_raw(path, body):
    """Never destroys an existing file. Returns (path, what_it_did)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        beside = path.with_name(path.name.replace('.json.gz', f'__refetch_{stamp}.json.gz'))
        with gzip.open(beside, 'wb') as fh:
            fh.write(body)
        return beside, 'WROTE_BESIDE'
    with gzip.open(path, 'wb') as fh:
        fh.write(body)
    return path, 'WROTE_NEW'

def load_raw(path):
    return json.loads(gzip.open(path, 'rb').read())

def summarise(body):
    """Row count and date range, read from a COPY. The saved file is never changed."""
    out = {'rows': None, 'date_min': None, 'date_max': None}
    try:
        data = json.loads(body)
    except Exception:
        return out
    rows = data if isinstance(data, list) else [data]
    out['rows'] = len(rows)
    ds = sorted(r['date'] for r in rows if isinstance(r, dict) and isinstance(r.get('date'), str))
    if ds:
        out['date_min'], out['date_max'] = ds[0], ds[-1]
    return out

# ---- block 0: the existing panel, from the repo's published manifest -------------
REPO_URL = 'https://github.com/ochofer/paper1-hazard-exposure-data.git'
if not Path('/content/repo').exists():
    os.system(f'git clone --depth 1 {REPO_URL} /content/repo')

panel = json.loads(Path('/content/repo/data/raw/manifest.json').read_text())['price_panel']
B0_ALL        = sorted({e['symbol'] for e in panel['per_symbol']})
B0_BENCHMARKS = sorted(panel['benchmark_symbols'])
B0_COMPANIES  = sorted(s for s in B0_ALL if s not in B0_BENCHMARKS)
assert len(B0_ALL) == 304 and len(B0_COMPANIES) == 302, (len(B0_ALL), len(B0_COMPANIES))

# ---- block 2: indices, ETFs, benchmarks -----------------------------------------
B2_SERIES = sorted(['^GSPC', '^SP500TR', 'SPY', 'IVV', 'AGG', 'BND', 'IEF', 'TLT',
                    'SHY', 'TIP', 'GLD', 'DBC', 'VNQ', 'EFA', 'EEM', 'VT'])

# Nine currencies of the existing panel. GBp is pence and ILA agorot: both hundredths.
CURRENCIES = ['AUD','CHF','DKK','EUR','GBp','ILA','NOK','SEK','USD']
MAJOR = {'GBp': 'GBP', 'ILA': 'ILS'}
FX_PAIRS = sorted({f"{MAJOR.get(c, c)}{b}" for c in CURRENCIES for b in ('USD','EUR')
                   if MAJOR.get(c, c) != b})

WINDOW_FROM = '1990-01-01'
WINDOW_TO   = datetime.now(timezone.utc).strftime('%Y-%m-%d')

print('block 0 :', len(B0_ALL), 'series =', len(B0_COMPANIES), 'companies +', B0_BENCHMARKS)
print('block 2 :', len(B2_SERIES), 'index/ETF series')
print('FX      :', len(FX_PAIRS), 'pairs')
print('window  :', WINDOW_FROM, 'to', WINDOW_TO)

### Block 1a: the S&P 500 universe

Downloads the current constituent list and the full constituent-change history, saves both
raw, then builds the union of every symbol that has ever appeared in either — as a current
member, as an addition, or as a removal.

**No filtering.** Odd-looking symbols stay in. A symbol dropped here is a symbol that
cannot be recovered after cancellation.

In [ ]:
def fetch_simple(path, dest_name, **params):
    """One request, saved raw. Used for the small reference pulls."""
    target = RAW / dest_name / 'all.json.gz'
    if target.exists() and target.stat().st_size > 0:
        print(f'  {dest_name}: already on disk, loaded from Drive')
        return load_raw(target)
    params['apikey'] = API_KEY
    r = requests.get(f'{BASE}/{path}', params=params, timeout=120)
    r.raise_for_status()
    save_raw(target, r.content)
    return r.json()

sp_current = fetch_simple('sp500-constituent', 'sp500_current')
sp_history = fetch_simple('historical-sp500-constituent', 'sp500_history')

print()
print('current list :', len(sp_current), 'rows; fields:', sorted(sp_current[0].keys()))
print('history      :', len(sp_history), 'rows; fields:', sorted(sp_history[0].keys()))

hist_dates = sorted(r['date'] for r in sp_history if r.get('date'))
print('history spans:', hist_dates[0], 'to', hist_dates[-1])

CURRENT = {r['symbol'].strip() for r in sp_current if r.get('symbol')}
ADDED   = {r['symbol'].strip() for r in sp_history
           if isinstance(r.get('symbol'), str) and r['symbol'].strip()}
REMOVED = {r['removedTicker'].strip() for r in sp_history
           if isinstance(r.get('removedTicker'), str) and r['removedTicker'].strip()}

B1_UNION = sorted(CURRENT | ADDED | REMOVED)

print()
print('current members      :', len(CURRENT))
print('distinct added       :', len(ADDED))
print('distinct removed     :', len(REMOVED))
print('UNION (block 1 list) :', len(B1_UNION))
print('  of which not currently in the index:', len(set(B1_UNION) - CURRENT))

(OUT / 'block1_union.txt').write_text('\n'.join(B1_UNION) + '\n')
print('\nunion written to', OUT / 'block1_union.txt')

### Block 4 list: Euronext Amsterdam

Only built if you reach block 4. Uses the company symbol list filtered to the `AMS`
exchange, whose suffix is `.AS`.

In [ ]:
B4_AMS = []
def build_ams():
    global B4_AMS
    syms = fetch_simple('company-symbols-list', 'company_symbols_list')
    B4_AMS = sorted({r['symbol'] for r in syms
                     if isinstance(r, dict) and str(r.get('exchange', '')).upper() in ('AMS',)
                     or (isinstance(r, dict) and str(r.get('symbol', '')).endswith('.AS'))})
    print('Euronext Amsterdam symbols:', len(B4_AMS))
    return B4_AMS

print('Not built yet. Call build_ams() when you reach block 4.')

---
## 4. What each block downloads

`applies` decides who gets what. Indices and ETFs are not companies: they get price-family
endpoints and a profile, never statements or market cap.

In [ ]:
ENDPOINTS = [
  # ---- block 0: the existing 304/302, tiers 1 and 2 ---------------------------
  dict(block='0', name='price_eod_full',          path='historical-price-eod/full',               applies='b0_all',  dated=True),
  dict(block='0', name='price_eod_dividend_adj',  path='historical-price-eod/dividend-adjusted',  applies='b0_all',  dated=True),
  dict(block='0', name='price_eod_non_split_adj', path='historical-price-eod/non-split-adjusted', applies='b0_all',  dated=True),
  dict(block='0', name='dividends',               path='dividends',                               applies='b0_co',   dated=False),
  dict(block='0', name='splits',                  path='splits',                                  applies='b0_co',   dated=False),
  dict(block='0', name='historical_market_cap',   path='historical-market-capitalization',        applies='b0_co',   dated=True),
  dict(block='0', name='shares_float',            path='shares-float',                            applies='b0_co',   dated=False),
  dict(block='0', name='profile',                 path='profile',                                 applies='b0_all',  dated=False),
  dict(block='0', name='delisted_companies',      path='delisted-companies',                      applies='once',    dated=False, paged=True),
  dict(block='0', name='fx_eod',                  path='historical-price-eod/full',               applies='fx',      dated=True),
  dict(block='0', name='income_statement_annual', path='income-statement',        applies='b0_co', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='0', name='income_statement_qtr',    path='income-statement',        applies='b0_co', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='0', name='balance_sheet_annual',    path='balance-sheet-statement', applies='b0_co', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='0', name='balance_sheet_qtr',       path='balance-sheet-statement', applies='b0_co', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='0', name='cash_flow_annual',        path='cash-flow-statement',     applies='b0_co', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='0', name='cash_flow_qtr',           path='cash-flow-statement',     applies='b0_co', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='0', name='key_metrics_annual',      path='key-metrics',             applies='b0_co', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='0', name='ratios_annual',           path='ratios',                  applies='b0_co', dated=False, extra=dict(period='annual',  limit=200)),

  # ---- block 2: indices, ETFs, treasury rates --------------------------------
  dict(block='2', name='b2_price_eod_full',         path='historical-price-eod/full',              applies='b2', dated=True),
  dict(block='2', name='b2_price_eod_dividend_adj', path='historical-price-eod/dividend-adjusted', applies='b2', dated=True),
  dict(block='2', name='b2_dividends',              path='dividends',                              applies='b2', dated=False),
  dict(block='2', name='b2_splits',                 path='splits',                                 applies='b2', dated=False),
  dict(block='2', name='b2_profile',                path='profile',                                applies='b2', dated=False),
  dict(block='2', name='treasury_rates',            path='treasury-rates',                         applies='once', dated=True),

  # ---- block 1c: the S&P 500 union -------------------------------------------
  dict(block='1c', name='b1_price_eod_full',         path='historical-price-eod/full',              applies='b1', dated=True),
  dict(block='1c', name='b1_price_eod_dividend_adj', path='historical-price-eod/dividend-adjusted', applies='b1', dated=True),
  dict(block='1c', name='b1_dividends',              path='dividends',                              applies='b1', dated=False),
  dict(block='1c', name='b1_splits',                 path='splits',                                 applies='b1', dated=False),
  dict(block='1c', name='b1_historical_market_cap',  path='historical-market-capitalization',       applies='b1', dated=True),
  dict(block='1c', name='b1_profile',                path='profile',                                applies='b1', dated=False),

  # ---- block 1d: delisted companies, full pagination -------------------------
  dict(block='1d', name='delisted_companies_paged',  path='delisted-companies',                     applies='paged'),

  # ---- block 3: fundamentals for the union -----------------------------------
  dict(block='3', name='b1_income_annual',   path='income-statement',        applies='b1', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='3', name='b1_income_qtr',      path='income-statement',        applies='b1', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='3', name='b1_balance_annual',  path='balance-sheet-statement', applies='b1', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='3', name='b1_balance_qtr',     path='balance-sheet-statement', applies='b1', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='3', name='b1_cashflow_annual', path='cash-flow-statement',     applies='b1', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='3', name='b1_cashflow_qtr',    path='cash-flow-statement',     applies='b1', dated=False, extra=dict(period='quarter', limit=400)),
  dict(block='3', name='b1_key_metrics',     path='key-metrics',             applies='b1', dated=False, extra=dict(period='annual',  limit=200)),
  dict(block='3', name='b1_ratios',          path='ratios',                  applies='b1', dated=False, extra=dict(period='annual',  limit=200)),

  # ---- block 4: Euronext Amsterdam, optional ---------------------------------
  dict(block='4', name='b4_price_eod_full',         path='historical-price-eod/full',              applies='b4', dated=True),
  dict(block='4', name='b4_price_eod_dividend_adj', path='historical-price-eod/dividend-adjusted', applies='b4', dated=True),
  dict(block='4', name='b4_profile',                path='profile',                                applies='b4', dated=False),
  dict(block='4', name='b4_income_annual',          path='income-statement',        applies='b4', dated=False, extra=dict(period='annual', limit=200)),
  dict(block='4', name='b4_balance_annual',         path='balance-sheet-statement', applies='b4', dated=False, extra=dict(period='annual', limit=200)),
  dict(block='4', name='b4_cashflow_annual',        path='cash-flow-statement',     applies='b4', dated=False, extra=dict(period='annual', limit=200)),

  # ---- block 0t3: tier 3 for the 302, last, if at all ------------------------
  dict(block='0t3', name='transcript_dates',       path='earning-call-transcript-dates', applies='b0_co', dated=False),
  dict(block='0t3', name='analyst_estimates',      path='analyst-estimates',   applies='b0_co', dated=False, extra=dict(period='annual', limit=200)),
  dict(block='0t3', name='price_target_consensus', path='price-target-consensus', applies='b0_co', dated=False),
  dict(block='0t3', name='esg_ratings',            path='esg-ratings',         applies='b0_co', dated=False),
  dict(block='0t3', name='vendor_dcf',             path='discounted-cash-flow', applies='b0_co', dated=False, sealed=True),
]

def targets_for(ep):
    return {'b0_all': B0_ALL, 'b0_co': B0_COMPANIES, 'b1': B1_UNION, 'b2': B2_SERIES,
            'b4': B4_AMS, 'fx': FX_PAIRS, 'once': [None], 'paged': [None]}[ep['applies']]

print(f"{'block':<7}{'requests':>10}")
for b in ('0', '1a', '2', '1c', '1d', '3', '4', '0t3'):
    n = sum(len(targets_for(e)) for e in ENDPOINTS if e['block'] == b)
    print(f'{b:<7}{n:>10}')
print(f"\ntotal excluding block 4 and 0t3: "
      f"{sum(len(targets_for(e)) for e in ENDPOINTS if e['block'] not in ('4','0t3')):,} requests")

---
## 5. Look before you loop

One request per endpoint, printing field names and date range.

**Which of these were verified against the live service on 7 September 2026** — field names
confirmed, not adjustment semantics:

`historical-price-eod/full`, `historical-market-capitalization`, `income-statement`,
`dividends`, `splits`, `profile`, `delisted-companies`, `sp500-constituent`,
`historical-sp500-constituent`, `treasury-rates`, `available-exchanges`.

Unverified and worth reading closely: `dividend-adjusted`, `non-split-adjusted`,
`shares-float`, `key-metrics`, `ratios`, `balance-sheet-statement`, `cash-flow-statement`,
`company-symbols-list`, and everything in block 0t3.

The sealed DCF endpoint prints field names and **not values**, on purpose.

In [ ]:
def probe(ep):
    tgts = targets_for(ep)
    if not tgts:
        print(f"{ep['name']:<30} (list not built yet)\n")
        return
    tgt = tgts[0]
    params = dict(ep.get('extra', {}))
    if tgt:
        params['symbol'] = tgt
    if ep.get('dated'):
        params['from'], params['to'] = WINDOW_FROM, WINDOW_TO
    status, data = try_once(ep['path'], **params)

    if status != 200 or data in (None, [], {}):
        print(f"{ep['name']:<30} DENIED or EMPTY  (status {status})\n")
        return
    rows = data if isinstance(data, list) else [data]
    first = rows[0] if rows else {}
    keys = sorted(first.keys()) if isinstance(first, dict) else type(first).__name__
    dates = sorted(r['date'] for r in rows if isinstance(r, dict) and isinstance(r.get('date'), str))
    span = f'{dates[0]} .. {dates[-1]}' if dates else 'no dates'
    print(f"{ep['name']:<30} OK  {len(rows):>6} rows  {span}")
    print(f"    fields: {keys}")
    if ep.get('sealed'):
        print('    [SEALED] values deliberately not shown')
    else:
        print(f"    first row: {json.dumps(first)[:200]}")
    print()

seen = set()
for ep in ENDPOINTS:
    if ep['path'] in seen:
        continue
    seen.add(ep['path'])
    probe(ep)
    time.sleep(0.4)

---
## 5b. Reconcile whatever is already on disk, before trusting any of it

The resume logic skips a file that exists and is non-empty. That is exactly the landmine
named in the 2 September handover-back: **a truncated file looks complete to an existence
check.** So before any loop runs, every file already on disk must be vouched for by a
ledger entry whose row count and date range match the file.

Anything that fails — no ledger entry at all, or an entry that disagrees with the file — is
**moved to `_quarantine/` and not skipped over**, so the loop refetches it. Nothing is
deleted.

On a first run there is nothing on disk and this costs nothing. Run it anyway: it is also
what proves that, on a resumed run, what you are keeping is real.

In [ ]:
QUARANTINE = OUT / '_quarantine'

def reconcile_with_ledger(verify_content=True):
    """Every raw file must be vouched for by a matching ledger entry, or it is quarantined."""
    if not LEDGER.exists():
        entries = {}
    else:
        entries = {}
        for line in LEDGER.read_text().splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            if str(r.get('disposition', '')).startswith('WROTE') and r.get('file'):
                entries[r['file']] = r

    on_disk = sorted(p for p in list(RAW.rglob('*.json.gz')) + list(VENDOR_DCF.rglob('*.json.gz'))
                     if QUARANTINE not in p.parents)
    print(f'files on disk: {len(on_disk)}   ledger entries: {len(entries)}')
    if not on_disk:
        print('Nothing on disk. Clean start, nothing to reconcile.')
        return {'checked': 0, 'kept': 0, 'quarantined': 0, 'reasons': {}}

    kept, moved, reasons = 0, [], defaultdict(int)
    for p in on_disk:
        rel = str(p.relative_to(OUT))
        e = entries.get(rel)
        why = None
        if e is None:
            why = 'no ledger entry'
        elif verify_content:
            try:
                body = gzip.open(p, 'rb').read()
            except Exception:
                why = 'unreadable gzip'
            else:
                m = summarise(body)
                if m['rows'] != e.get('rows'):
                    why = f"row count {m['rows']} != ledger {e.get('rows')}"
                elif m['date_min'] != e.get('date_min') or m['date_max'] != e.get('date_max'):
                    why = 'date range disagrees with ledger'
        if why:
            dest = QUARANTINE / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                dest = dest.with_name(dest.name.replace(
                    '.json.gz', f"__{datetime.now(timezone.utc).strftime('%H%M%S')}.json.gz"))
            p.rename(dest)
            moved.append((rel, why)); reasons[why.split(' ')[0]] += 1
        else:
            kept += 1

    print(f'kept (vouched for): {kept}')
    print(f'quarantined       : {len(moved)}')
    for rel, why in moved[:30]:
        print(f'   {rel}  <- {why}')
    if len(moved) > 30:
        print(f'   ...and {len(moved)-30} more')
    if moved:
        (QUARANTINE / 'WHY.md').write_text(
            '# Quarantined, not deleted\n\n'
            'These files were on disk but no ledger entry vouched for them, or the entry\n'
            'disagreed with the file. A truncated file looks complete to an existence check,\n'
            'so they are moved aside and refetched rather than skipped.\n\n'
            + '\n'.join(f'- `{r}` — {w}' for r, w in moved) + '\n')
        print(f'\nreasons written to {QUARANTINE / "WHY.md"}')
    return {'checked': len(on_disk), 'kept': kept, 'quarantined': len(moved),
            'reasons': dict(reasons)}

recon = reconcile_with_ledger()

---
## 6. The download

**Start conservative.** Being rate-limited into a ban before cancellation is the one
mistake with no recovery. Look up your plan's requests-per-minute limit and put *half* of
it in `CALLS_PER_MINUTE`.

In [ ]:
CALLS_PER_MINUTE = 150        # half your plan's documented limit for the first run
MIN_GAP          = 60.0 / CALLS_PER_MINUTE
MAX_RETRIES      = 6
STOP_AFTER_429S  = 5

_last, _c429 = [0.0], [0]

def fetch(path, params):
    """One request, with polite waiting and backoff. Returns (status, raw bytes)."""
    params = dict(params)
    params['apikey'] = API_KEY
    for attempt in range(MAX_RETRIES):
        gap = time.monotonic() - _last[0]
        if gap < MIN_GAP:
            time.sleep(MIN_GAP - gap)
        _last[0] = time.monotonic()
        try:
            r = requests.get(f'{BASE}/{path}', params=params, timeout=90)
        except Exception:
            time.sleep(5 * (2 ** attempt)); continue
        if r.status_code == 429:
            _c429[0] += 1
            if _c429[0] >= STOP_AFTER_429S:
                raise SystemExit('Five rate-limit responses in a row. Stopping on purpose.\n'
                                 'Lower CALLS_PER_MINUTE and run again - nothing is lost.')
            wait = min(600, 5 * (2 ** attempt)) + random.uniform(0, 3)
            print(f'    rate limited, waiting {wait:.0f}s'); time.sleep(wait); continue
        _c429[0] = 0
        if r.status_code >= 500:
            time.sleep(min(600, 5 * (2 ** attempt))); continue
        return r.status_code, r.content
    return 0, b''

BLOCK_STATS = {}          # block -> {requests, minutes}; the handover-back needs these

def run_blocks(blocks):
    counts = defaultdict(int)
    started = time.time()
    ledger = LEDGER.open('a')
    for ep in [e for e in ENDPOINTS if e['block'] in blocks]:
        if ep['applies'] == 'paged':
            run_paged(ep, ledger, counts); continue
        tgts = targets_for(ep)
        if not tgts:
            print(f"\n=== {ep['name']}: target list empty, skipped ==="); continue
        root = (VENDOR_DCF if ep.get('sealed') else RAW) / ep['name']
        print(f"\n=== block {ep['block']}  {ep['name']}  ({len(tgts)} requests) ===", flush=True)
        for i, sym in enumerate(tgts, 1):
            target = root / f"{sanitise(sym or 'all')}.json.gz"
            if target.exists() and target.stat().st_size > 0:
                counts['SKIPPED'] += 1; continue
            params = dict(ep.get('extra', {}))
            if sym:
                params['symbol'] = sym
            if ep.get('dated'):
                params['from'], params['to'] = WINDOW_FROM, WINDOW_TO
            status, body = fetch(ep['path'], params)
            if status != 200 or not body:
                counts['FAILED'] += 1
                ledger.write(json.dumps({'block': ep['block'], 'endpoint': ep['name'],
                    'symbol': sym, 'http_status': status, 'disposition': 'FAILED',
                    'fetched_utc': datetime.now(timezone.utc).isoformat()}) + '\n')
                ledger.flush(); continue
            if body.strip() in (b'[]', b'{}'):
                counts['EMPTY'] += 1
            path, how = save_raw(target, body)
            counts[how] += 1
            meta = summarise(body)
            ledger.write(json.dumps({
                'block': ep['block'], 'endpoint': ep['name'], 'symbol': sym, 'path': ep['path'],
                'params': {k: v for k, v in params.items() if k != 'apikey'},
                'fetched_utc': datetime.now(timezone.utc).isoformat(),
                'http_status': status, 'bytes': len(body),
                'sha256': hashlib.sha256(body).hexdigest(),
                'file': str(path.relative_to(OUT)), 'disposition': how,
                'sealed': bool(ep.get('sealed')), **meta}) + '\n')
            ledger.flush()
            if i % 100 == 0 or i == len(tgts):
                print(f"  [{i}/{len(tgts)}] {sym}  rows={meta['rows']}  "
                      f"{meta['date_min']}..{meta['date_max']}", flush=True)
    ledger.close()
    mins = (time.time() - started) / 60
    requests_made = sum(counts[k] for k in ('WROTE_NEW', 'WROTE_BESIDE', 'FAILED'))
    for b in blocks:
        BLOCK_STATS[b] = {'requests_made': requests_made, 'minutes': round(mins, 1),
                          'skipped_already_on_disk': counts.get('SKIPPED', 0),
                          'counts': dict(counts)}
    (OUT / 'block_stats.json').write_text(json.dumps(BLOCK_STATS, indent=2))
    print('\n' + json.dumps(dict(counts), indent=2))
    print(f'requests actually made: {requests_made}   elapsed: {mins:.1f} min')
    return dict(counts), mins

def run_paged(ep, ledger, counts):
    """delisted-companies, walked to exhaustion. One file per page."""
    root = RAW / ep['name']
    print(f"\n=== block {ep['block']}  {ep['name']} (paginated) ===", flush=True)
    page = 0
    while True:
        target = root / f'page_{page:04d}.json.gz'
        if target.exists() and target.stat().st_size > 0:
            page += 1; counts['SKIPPED'] += 1; continue
        status, body = fetch(ep['path'], {'page': page, 'limit': 100})
        if status != 200 or not body:
            counts['FAILED'] += 1; print(f'  page {page}: HTTP {status}, stopping'); break
        try:
            rows = json.loads(body)
        except Exception:
            rows = []
        if not rows:
            print(f'  page {page}: empty, pagination complete ({page} pages)'); break
        path, how = save_raw(target, body)
        counts[how] += 1
        ledger.write(json.dumps({'block': ep['block'], 'endpoint': ep['name'],
            'symbol': f'page_{page:04d}', 'path': ep['path'], 'params': {'page': page, 'limit': 100},
            'fetched_utc': datetime.now(timezone.utc).isoformat(), 'http_status': status,
            'bytes': len(body), 'sha256': hashlib.sha256(body).hexdigest(),
            'file': str(path.relative_to(OUT)), 'disposition': how, 'rows': len(rows)}) + '\n')
        ledger.flush()
        if page % 20 == 0:
            print(f'  page {page}: {len(rows)} rows', flush=True)
        page += 1

print('Ready.', CALLS_PER_MINUTE, 'requests per minute.')

### Run the blocks, in this order

Run one cell at a time and read the counts before moving on. **If Colab disconnects, run
the same cell again** — it skips what is already in Drive.

Record the printed counts and elapsed minutes for each block; the handover needs them.

In [ ]:
# Block 0: finish the existing 304/302, tiers 1 and 2.
counts_b0, mins_b0 = run_blocks(['0'])

In [ ]:
# Block 2: indices, ETFs, treasury rates. Small - do it early so a timeout cannot lose it.
counts_b2, mins_b2 = run_blocks(['2'])

In [ ]:
# Block 1c + 1d: the S&P 500 union, then the delisted list.
counts_b1, mins_b1 = run_blocks(['1c', '1d'])

In [ ]:
# Block 3: fundamentals for the union. First to drop if time is short.
counts_b3, mins_b3 = run_blocks(['3'])

In [ ]:
# Block 4, OPTIONAL: Euronext Amsterdam. Only if everything above is done.
build_ams()
counts_b4, mins_b4 = run_blocks(['4'])

In [ ]:
# Block 0t3, LAST and optional: tier 3 for the 302. Paper 1's event study is shelved.
counts_t3, mins_t3 = run_blocks(['0t3'])

---
## 7. The manifest

Writes `manifest__fmp_pull_<timestamp>.json` beside the data: per endpoint and symbol, the
UTC fetch time, row count, first and last date, and the SHA-256 of the raw file.

It does **not** touch `data/raw/manifest.json` in the repo. Nothing from this pull is ever
committed.

The SHA-256 recorded here is of the response body as received. The verification script you
run on your own machine re-hashes the file on disk and compares, which is what proves the
local copy is intact.

In [ ]:
rows = [json.loads(l) for l in LEDGER.read_text().splitlines() if l.strip()]
good = {(r['endpoint'], r.get('symbol')): r for r in rows
        if str(r.get('disposition', '')).startswith('WROTE')}

files = []
for (ep, sym), r in sorted(good.items(), key=lambda kv: (kv[0][0], str(kv[0][1]))):
    files.append({k: r.get(k) for k in
                  ('block', 'endpoint', 'symbol', 'file', 'sha256', 'bytes',
                   'rows', 'date_min', 'date_max', 'fetched_utc', 'params')})

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
manifest = {
    'schema': 'fmp-pull-manifest/1',
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'pull': {
        'source': 'Financial Modeling Prep, /stable endpoints',
        'window_requested': {'from': WINDOW_FROM, 'to': WINDOW_TO},
        'licence': 'Retained under FMP written confirmation: personal use, local computer, '
                   'never published. Not redistributable. Never committed to git.',
    },
    'universes': {
        'block0_series': B0_ALL, 'block0_companies': len(B0_COMPANIES),
        'block0_benchmarks': B0_BENCHMARKS,
        'block1_union_size': len(B1_UNION), 'block1_union': B1_UNION,
        'block2_series': B2_SERIES, 'block4_ams_size': len(B4_AMS),
        'fx_pairs': FX_PAIRS,
    },
    'counts': {'files': len(files),
               'bytes': sum(f['bytes'] or 0 for f in files),
               'ledger_entries': len(rows)},
    'files': files,
}

mpath = OUT / f'manifest__fmp_pull_{stamp}.json'
mpath.write_text(json.dumps(manifest, indent=2))
mhash = hashlib.sha256(mpath.read_bytes()).hexdigest()
(OUT / f'manifest__fmp_pull_{stamp}.json.sha256').write_text(f'{mhash}  {mpath.name}\n')

print('manifest :', mpath.name)
print('files    :', len(files))
print('SHA-256  :', mhash)
print('\nRecord that filename and hash in the handover.')

---
## 8. Section C checks

Seven checks. Their numbers go into the handover, and Carlo does not cancel until the
design chat has read them. Each prints what it found and does not repair anything.

In [ ]:
# ---- helpers ---------------------------------------------------------------
def raw_path(endpoint, sym):
    p = RAW / endpoint / f'{sanitise(sym)}.json.gz'
    return p if p.exists() else None

def rows_for(endpoint, sym):
    p = raw_path(endpoint, sym)
    if not p:
        return None
    try:
        d = load_raw(p)
    except Exception:
        return None
    return d if isinstance(d, list) else [d]

def price_dates(endpoint, sym):
    r = rows_for(endpoint, sym)
    if not r:
        return []
    return sorted(x['date'] for x in r if isinstance(x, dict) and isinstance(x.get('date'), str))

REPORT = []
def say(s=''):
    print(s)
    REPORT.append(s)

In [ ]:
# ---- C1: constituent history ------------------------------------------------
say('## C1. Constituent history')
say()
hist_dates = sorted(r['date'] for r in sp_history if r.get('date'))
say(f'- earliest change date: {hist_dates[0]}')
say(f'- latest change date:   {hist_dates[-1]}')
say(f'- change rows:          {len(sp_history)}')
say(f'- distinct symbols in the union: {len(B1_UNION)}')
say()

per_year = defaultdict(int)
for d in hist_dates:
    per_year[d[:4]] += 1
say('### Change rows per year, 1957 to 2026')
say()
say('Every year in the range, including the empty ones. A year with no rows is a year the')
say('history does not cover, which matters more than the reconstructed count below.')
say()
Y0, Y1 = 1957, int(hist_dates[-1][:4])
say('| year | rows | note |')
say('|---|---:|---|')
for y in range(Y0, Y1 + 1):
    k = str(y)
    n = per_year.get(k, 0)
    if y == 1957:
        note = '**founding snapshot, not changes**'
    elif n == 0:
        note = 'no coverage'
    else:
        note = ''
    say(f'| {y} | {n} | {note} |')
say()
zero_years = [str(y) for y in range(Y0, Y1 + 1) if per_year.get(str(y), 0) == 0]
say(f'- years with no recorded changes: **{len(zero_years)}** '
    f'({", ".join(zero_years) if zero_years else "none"})')
first_unbroken = None
for y in range(Y0 + 1, Y1 + 1):
    if all(per_year.get(str(k), 0) > 0 for k in range(y, Y1 + 1)):
        first_unbroken = y
        break
say(f'- first year from which every later year has at least one recorded change: '
    f'**{first_unbroken}**')
say()
say('### The density leg, under both versions')
say()
say('The design chat replaced a per-year floor with a rolling-window test on 7 September,')
say('before C2 and C3 existed, because a per-year floor cannot tell a quiet year from a')
say('gap. Both are reported here. Which governs is the design chat\'s to record, and the')
say('reason is the mis-specification, never the year either version produces.')
say()

# superseded: at least 10 rows in the year, and every year after
first_ten = next((y for y in range(Y0 + 1, Y1 + 1)
                  if all(per_year.get(str(k), 0) >= 10 for k in range(y, Y1 + 1))), None)

# governing: every rolling three-calendar-year window holds >= 24 rows, and no year is zero
def density_ok(start):
    if any(per_year.get(str(y), 0) == 0 for y in range(start, Y1 + 1)):
        return False
    for y in range(start, Y1 - 1):
        if sum(per_year.get(str(k), 0) for k in (y, y + 1, y + 2)) < 24:
            return False
    return True

first_roll = next((y for y in range(Y0, Y1 + 1) if density_ok(y)), None)

say('| version | test | year it yields |')
say('|---|---|---|')
say(f'| superseded | at least 10 rows in the year, and in every later year | **{first_ten}** |')
say(f'| **governing** | every rolling 3-year window holds at least 24 rows, and no year '
    f'is zero | **{first_roll}** |')
say()
say('The superseded version turned on a single row: 2003 has 9, one short of 10. The')
say('governing version passes it, because 2003 to 2005 holds 47.')
say()
say('Boundary of the governing test, so the answer can be checked by eye:')
say()
say('| window | rows | verdict |')
say('|---|---:|---|')
for y in range(max(Y0, first_roll - 4), first_roll + 4):
    w = sum(per_year.get(str(k), 0) for k in (y, y + 1, y + 2))
    say(f'| {y}–{y+2} | {w} | {"fails" if w < 24 else "passes"} |')
say()
say(f'- years holding zero rows: {", ".join(zero_years) if zero_years else "none"} — all')
say(f'  before {first_roll}, which is what closes the question.')
say()
say('This is one leg of four. The governing window is the intersection with the year-end')
say('count leg and with C2 and C3 by decade, and it is set at the handover-back.')
say()

members = set(CURRENT)
rows_sorted = sorted(
    [(r['date'], (r.get('symbol') or '').strip(), (r.get('removedTicker') or '').strip())
     for r in sp_history if r.get('date')], reverse=True)
i = 0
target_n = len(CURRENT)
say(f'Reconstructed membership at each year-end, working backward from the current '
    f'list of {target_n}.')
say()
say('| year-end | members | within 10 of current |')
say('|---|---:|---|')
first_ok = None
for year in range(int(hist_dates[-1][:4]), 1989, -1):
    ye = f'{year}-12-31'
    while i < len(rows_sorted) and rows_sorted[i][0] > ye:
        _, add, rem = rows_sorted[i]
        if add: members.discard(add)
        if rem: members.add(rem)
        i += 1
    n = len(members)
    ok = abs(n - target_n) <= 10
    if ok:
        first_ok = year
    say(f'| {year} | {n} | {"yes" if ok else "no"} |')
say()
say(f'**Usable backtest window starts at year-end {first_ok}** by the stated rule.')
say()
say('Read that with the per-year table. The count staying near 500 going backward is')
say('necessary but not sufficient: if change rows are missing, fewer changes get undone')
say('and the count stays near the current size for that reason rather than because the')
say('membership is right. The per-year counts are the honest guide to where the history')
say('is real. History not repaired, per instruction.')
say()

In [ ]:
# ---- C1b: does the current list agree with the history? ---------------------
# The whole C1 reconstruction is anchored on the current list. If that anchor is
# wrong, every year-end count inherits the error. This tests the anchor.
say('## C1b. Current list against the history')
say()
last_add, last_rem = defaultdict(str), defaultdict(str)
for r in sp_history:
    d = r.get('date') or ''
    a = (r.get('symbol') or '').strip()
    rm = (r.get('removedTicker') or '').strip()
    if a and d > last_add[a]:  last_add[a] = d
    if rm and d > last_rem[rm]: last_rem[rm] = d

omissions = sorted(s for s in set(last_add)
                   if last_add[s] > last_rem.get(s, '') and s not in CURRENT)
stale     = sorted(s for s in CURRENT if last_rem.get(s, '') > last_add.get(s, ''))
never     = sorted(s for s in CURRENT if s not in last_add and s not in last_rem)

say(f'- current list size: {len(CURRENT)}')
say(f'- history says still a member, but absent from the current list: **{len(omissions)}**')
for s in omissions:
    nm = next((r.get('addedSecurity') for r in sp_history
               if (r.get('symbol') or '').strip() == s), '')
    say(f'  - `{s}` last added {last_add[s]} — {nm}')
say(f'- in the current list but the history\'s last event for it was a removal: **{len(stale)}**')
for s in stale:
    say(f'  - `{s}` removed {last_rem[s]}, last addition recorded {last_add.get(s, "never")}')
say(f'- in the current list but absent from the history entirely: **{len(never)}**')
say(f'  {", ".join(never)}')
say()
say('**These are not missing companies. They are the same companies under two ticker')
say('vocabularies.** The two files do not agree with each other, and a point-in-time')
say('membership series built naively from them will be wrong in these named places.')
say()

# ---- the reconciliation table, per the design chat's instruction -------------
# Facts only. No pairing is asserted: CIK and ISIN are the keys Project B's data-layer
# notebook will reconcile on, and that reconciliation is explicitly not done here.
def profile_keys(sym):
    for ep in ('b1_profile', 'profile', 'b2_profile'):
        r = rows_for(ep, sym)
        if r:
            p = r[0] if isinstance(r, list) else r
            if isinstance(p, dict):
                return (p.get('cik') or '', p.get('isin') or '',
                        p.get('companyName') or p.get('name') or '')
    return ('', '', '')

say('### Reconciliation table')
say()
say('Facts only, no judgement attached. CIK and ISIN come from the pulled profiles and are')
say('the keys to reconcile on. The pairing itself belongs to Project B\'s data-layer')
say('notebook, not to extraction.')
say()
say('| symbol | in current list | in history | last history event | date | CIK | ISIN | name |')
say('|---|---|---|---|---|---|---|---|')
for s in sorted(set(omissions) | set(stale) | set(never)):
    in_cur = 'yes' if s in CURRENT else 'no'
    la, lr = last_add.get(s, ''), last_rem.get(s, '')
    if not la and not lr:
        in_hist, ev, when = 'no', 'never appears', ''
    elif lr > la:
        in_hist, ev, when = 'yes', 'removed', lr
    else:
        in_hist, ev, when = 'yes', 'added', la
    cik, isin, name = profile_keys(s)
    say(f'| `{s}` | {in_cur} | {in_hist} | {ev} | {when} | {cik} | {isin} | {name[:34]} |')
say()
say(f'{len(set(omissions) | set(stale) | set(never))} securities. C7 may add more; any it')
say('finds are appended to this table rather than reported separately.')
say()
say('The `stale` group differs in kind from the other two: those are re-additions the')
say('history never recorded, so a reconstruction drops them decades early rather than')
say('mislabelling them.')
say()
say('Not repaired, per instruction.')
say()

In [ ]:
# ---- C2: removed tickers ----------------------------------------------------
say('## C2. Removed tickers')
say()
removal_date = {}
for r in sp_history:
    t = (r.get('removedTicker') or '').strip()
    if t and r.get('date'):
        # keep the most recent removal for a ticker
        if t not in removal_date or r['date'] > removal_date[t]:
            removal_date[t] = r['date']

# "10 trading days" is measured against a real calendar, not approximated from weekdays.
gspc = price_dates('b2_price_eod_full', '^GSPC')
cal = gspc or price_dates('b2_price_eod_full', 'SPY')
cal_name = '^GSPC' if gspc else ('SPY' if cal else None)
if cal:
    cal_idx = {d: i for i, d in enumerate(cal)}
    say(f'Trading calendar: {cal_name}, {len(cal)} sessions.')
else:
    cal_idx = {}
    say('No index series on disk; falling back to calendar days (14 days ~ 10 sessions).')
say()

def sessions_between(a, b):
    if cal_idx:
        ia = next((cal_idx[d] for d in cal if d >= min(a, b)), None)
        ib = next((cal_idx[d] for d in cal if d >= max(a, b)), None)
        if ia is not None and ib is not None:
            return abs(ib - ia)
    return abs((date.fromisoformat(b) - date.fromisoformat(a)).days) * 5 // 7

removed_sorted = sorted(removal_date)
with_prices, within10 = 0, 0
missing = []
for t in removed_sorted:
    ds = price_dates('b1_price_eod_full', t)
    if not ds:
        missing.append(t); continue
    with_prices += 1
    if sessions_between(ds[-1], removal_date[t]) <= 10:
        within10 += 1

checked = len(removed_sorted)
say(f'- removed tickers in the history: {checked}')
say(f'- with any price rows: {with_prices} ({100*with_prices/max(1,checked):.1f}%)')
say(f'- whose last price is within 10 trading days of removal: '
    f'{within10} ({100*within10/max(1,with_prices):.1f}% of those with prices)')
say()

# ---- by decade, for the governing-window rule -------------------------------
say('### C2 by decade of removal')
say()
say('The third leg of the governing-window rule: the share of a decade\'s removed tickers')
say('whose last price sits within 10 trading days of removal must be at least 90%.')
say()
dec = defaultdict(lambda: {'n': 0, 'priced': 0, 'near': 0})
for t in removed_sorted:
    d = f"{removal_date[t][:3]}0s"
    dec[d]['n'] += 1
    ds = price_dates('b1_price_eod_full', t)
    if not ds:
        continue
    dec[d]['priced'] += 1
    if sessions_between(ds[-1], removal_date[t]) <= 10:
        dec[d]['near'] += 1
say('| decade | removed | with prices | % | last price within 10 sessions | % of priced | >=90% |')
say('|---|---:|---:|---:|---:|---:|---|')
C2_BY_DECADE = {}
for d in sorted(dec):
    v = dec[d]
    p_pct = 100 * v['priced'] / max(1, v['n'])
    n_pct = 100 * v['near'] / max(1, v['priced'])
    C2_BY_DECADE[d] = n_pct
    say(f"| {d} | {v['n']} | {v['priced']} | {p_pct:.1f}% | {v['near']} | {n_pct:.1f}% | "
        f"{'yes' if n_pct >= 90 else 'no'} |")
say(f'- with no price rows at all: {len(missing)}')
if missing:
    say(f'  {", ".join(missing[:60])}' + (' ...' if len(missing) > 60 else ''))
say()

In [ ]:
# ---- C3: market-cap coverage against membership months ----------------------
say('## C3. Market-cap coverage')
say()
def membership_months(sym):
    """Months the symbol was in the index, CLAMPED TO THE PULL WINDOW.

    The clamp matters. Symbols in the 1957 founding block have no recorded addition,
    so their span would otherwise open at 1957-03 and the >=90% test would demand
    market-cap rows back to 1957, which were never requested. Clamping to
    WINDOW_FROM asks the honest question: does market cap cover the membership
    months we actually asked for? Months before the window are reported separately
    rather than silently counted as misses.
    """
    adds = sorted(r['date'] for r in sp_history
                  if (r.get('symbol') or '').strip() == sym and r.get('date'))
    rems = sorted(r['date'] for r in sp_history
                  if (r.get('removedTicker') or '').strip() == sym and r.get('date'))
    spans = []
    if sym in CURRENT:
        spans.append((adds[-1] if adds else hist_dates[0], WINDOW_TO))
    for rem in rems:
        prior = [a for a in adds if a < rem]
        spans.append((prior[-1] if prior else hist_dates[0], rem))
    months, pre_window = set(), 0
    for a, b in spans:
        if b < WINDOW_FROM:
            pre_window += 1
            continue
        a = max(a, WINDOW_FROM)
        ya, ma = int(a[:4]), int(a[5:7])
        yb, mb = int(b[:4]), int(b[5:7])
        while (ya, ma) <= (yb, mb):
            months.add(f'{ya:04d}-{ma:02d}')
            ma += 1
            if ma > 12: ma, ya = 1, ya + 1
    return months, pre_window

ok90, assessed, no_mcap, no_span, wholly_pre = 0, 0, [], [], []
for sym in B1_UNION:
    need, pre = membership_months(sym)
    if not need:
        (wholly_pre if pre else no_span).append(sym)
        continue
    assessed += 1
    r = rows_for('b1_historical_market_cap', sym)
    if not r:
        no_mcap.append(sym); continue
    have = {x['date'][:7] for x in r if isinstance(x, dict) and isinstance(x.get('date'), str)}
    if len(need & have) / len(need) >= 0.90:
        ok90 += 1

say(f'- union symbols: {len(B1_UNION)}')
say(f'- assessed (membership months inside the {WINDOW_FROM} window): {assessed}')
say(f'- **market-cap rows covering >=90% of membership months: {ok90} '
    f'({100*ok90/max(1,assessed):.1f}% of assessed)**')
say(f'- assessed but no market-cap file or no rows: {len(no_mcap)}')
if no_mcap:
    say(f'  {", ".join(no_mcap[:60])}' + (' ...' if len(no_mcap) > 60 else ''))
say()
say()
# ---- by decade, for the governing-window rule -------------------------------
say('### C3 by decade of membership')
say()
say("The fourth leg of the governing-window rule: the share of a decade's members whose")
say('market-cap rows cover at least 90% of their membership months **in that decade** must')
say('itself be at least 90%. A symbol counts in every decade it was a member.')
say()
dec3 = defaultdict(lambda: {'n': 0, 'ok': 0})
for sym in B1_UNION:
    need, _ = membership_months(sym)
    if not need:
        continue
    r = rows_for('b1_historical_market_cap', sym) or []
    have = {x['date'][:7] for x in r if isinstance(x, dict) and isinstance(x.get('date'), str)}
    by_dec = defaultdict(set)
    for m in need:
        by_dec[f'{m[:3]}0s'].add(m)
    for d, months in by_dec.items():
        dec3[d]['n'] += 1
        if len(months & have) / len(months) >= 0.90:
            dec3[d]['ok'] += 1
say('| decade | members | market cap covers >=90% of that decade | % | >=90% |')
say('|---|---:|---:|---:|---|')
C3_BY_DECADE = {}
for d in sorted(dec3):
    v = dec3[d]
    pct = 100 * v['ok'] / max(1, v['n'])
    C3_BY_DECADE[d] = pct
    say(f"| {d} | {v['n']} | {v['ok']} | {pct:.1f}% | {'yes' if pct >= 90 else 'no'} |")
say()
say('The design chat sets the governing window after the pull, from these two tables plus')
say('the reconstructed counts and the change-row counts. All four legs must hold for a year')
say('and every year after it. Nothing here chooses the window.')
say()

say(f'- excluded, membership ended before {WINDOW_FROM}: {len(wholly_pre)}')
say(f'- excluded, no determinable membership span: {len(no_span)}')
say('  These are symbols that appear as an addition but never as a removal and are not')
say('  current — the signature of a rename (FB to META). They are counted in C7, not here,')
say('  because a denominator cannot be built for them. Listed so the exclusion is visible:')
if no_span:
    for j in range(0, min(len(no_span), 120), 12):
        say('    ' + '  '.join(no_span[j:j+12]))
    if len(no_span) > 120:
        say(f'    ...and {len(no_span)-120} more')
say()

In [ ]:
# ---- C4: three known splits, and which series shows the step ----------------
say('## C4. Known splits, and which price series is split-adjusted')
say()
KNOWN = [('AAPL', '2020-08-31', '4-for-1'),
         ('TSLA', '2022-08-25', '3-for-1'),
         ('NVDA', '2024-06-10', '10-for-1')]

def step_ratio(endpoint, sym, day):
    """close on the last bar before `day` divided by close on `day`."""
    r = rows_for(endpoint, sym)
    if not r:
        return None
    bars = {x['date']: x for x in r if isinstance(x, dict) and x.get('date')}
    ds = sorted(bars)
    after = [d for d in ds if d >= day]
    before = [d for d in ds if d < day]
    if not after or not before:
        return None
    try:
        return bars[before[-1]]['close'] / bars[after[0]]['close']
    except Exception:
        return None

say('| symbol | in raw splits? | ratio recorded | full series step | dividend-adj step |')
say('|---|---|---|---:|---:|')
for sym, day, label in KNOWN:
    sp = rows_for('b1_splits', sym) or rows_for('splits', sym) or []
    hit = [x for x in sp if isinstance(x, dict) and x.get('date') == day]
    ratio = (f"{hit[0].get('numerator')}:{hit[0].get('denominator')}" if hit else '-')
    f_step = step_ratio('b1_price_eod_full', sym, day)
    d_step = step_ratio('b1_price_eod_dividend_adj', sym, day)
    fs = f'{f_step:.2f}' if f_step is not None else '-'
    ds = f'{d_step:.2f}' if d_step is not None else '-'
    say(f'| {sym} {label} {day} | {"yes" if hit else "NO"} | {ratio} | {fs} | {ds} |')
say()
say('A step near 1.00 means the series is already split-adjusted; a step near the split')
say('ratio means it is not. PROJECT_STATUS records that historical-price-eod/full is')
say('split-adjusted but NOT dividend-adjusted. This check tests that claim rather than')
say('assuming it: the 2 September probe verified field names, not adjustment semantics.')
say()

In [ ]:
# ---- C5: three acquired names ----------------------------------------------
say('## C5. Acquired names: last price against removal date')
say()
acq = [r for r in sp_history
       if (r.get('removedTicker') or '').strip()
       and 'acqui' in str(r.get('reason', '')).lower()]
acq.sort(key=lambda r: r['date'], reverse=True)
picked, seen_t = [], set()
for r in acq:
    t = r['removedTicker'].strip()
    if t in seen_t:
        continue
    seen_t.add(t); picked.append(r)
    if len(picked) == 3:
        break

say('| ticker | removed | last price | gap (days) | reason |')
say('|---|---|---|---:|---|')
for r in picked:
    t = r['removedTicker'].strip()
    ds = price_dates('b1_price_eod_full', t)
    last = ds[-1] if ds else None
    gap = ((date.fromisoformat(r['date']) - date.fromisoformat(last)).days
           if last else None)
    say(f'| {t} | {r["date"]} | {last or "none"} | {gap if gap is not None else "-"} | '
        f'{str(r.get("reason",""))[:70]} |')
say()

In [ ]:
# ---- C6: SPY against ^SP500TR ----------------------------------------------
say('## C6. SPY against ^SP500TR, annual total return')
say()
def annual_returns(endpoint, sym):
    r = rows_for(endpoint, sym)
    if not r:
        return {}
    bars = {x['date']: x.get('close') for x in r
            if isinstance(x, dict) and x.get('date') and x.get('close')}
    out = {}
    for y in range(2015, 2026):
        ds = sorted(d for d in bars if d.startswith(str(y)))
        if len(ds) > 200:
            out[y] = bars[ds[-1]] / bars[ds[0]] - 1
    return out

spy = annual_returns('b2_price_eod_dividend_adj', 'SPY')
tr  = annual_returns('b2_price_eod_full', '^SP500TR')
if not spy or not tr:
    say('Not computable: one of the two series is missing. '
        f'SPY years={len(spy)}, ^SP500TR years={len(tr)}')
else:
    say('| year | SPY (div-adj) | ^SP500TR | difference (bp) |')
    say('|---|---:|---:|---:|')
    diffs = []
    for y in sorted(set(spy) & set(tr)):
        d = (spy[y] - tr[y]) * 10000
        diffs.append(abs(d))
        say(f'| {y} | {spy[y]*100:.2f}% | {tr[y]*100:.2f}% | {d:+.0f} |')
    say()
    say(f'- median absolute difference: {sorted(diffs)[len(diffs)//2]:.0f} bp')
    say('A few tens of basis points is expected (ETF fee and tracking). Hundreds means')
    say('one series is not what it claims to be.')
say()

In [ ]:
# ---- C7: symbols with no profile and no prices ------------------------------
say('## C7. Symbols in the history with neither profile nor prices')
say()
ghosts = []
for sym in B1_UNION:
    has_p = bool(rows_for('b1_profile', sym))
    has_x = bool(price_dates('b1_price_eod_full', sym))
    if not has_p and not has_x:
        ghosts.append(sym)
say(f'- {len(ghosts)} of {len(B1_UNION)} union symbols returned neither')
say('These are where renames hide (FB to META style). Listed, not repaired.')
say()
if ghosts:
    for j in range(0, len(ghosts), 12):
        say('    ' + '  '.join(ghosts[j:j+12]))
say()

report_path = OUT / f'SECTION_C_CHECKS_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.md'
if report_path.exists():
    report_path = report_path.with_name(
        report_path.stem + datetime.now(timezone.utc).strftime('__%H%M%SZ') + '.md')
report_path.write_text('\n'.join(REPORT))
print('\n\nSection C written to:', report_path)

---
## 9. Completeness report

Coverage per endpoint against what was requested, date ranges obtained, and what the plan
refused. This plus section C is what the design chat reads before Carlo cancels.

In [ ]:
rows = [json.loads(l) for l in LEDGER.read_text().splitlines() if l.strip()]
good = {(r['endpoint'], r.get('symbol')): r for r in rows
        if str(r.get('disposition', '')).startswith('WROTE')}
by_ep = defaultdict(dict)
for (ep, sym), r in good.items():
    by_ep[ep][sym] = r

L = [f'# Completeness report - {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")} UTC', '']
L += ['| block | endpoint | got | empty | missing | of |', '|---|---|---:|---:|---:|---:|']
gaps = []
for ep in ENDPOINTS:
    if ep['applies'] in ('once', 'paged'):
        continue
    tgts, got = targets_for(ep), by_ep.get(ep['name'], {})
    if not tgts:
        continue
    usable  = [s for s in tgts if got.get(s) and (got[s].get('rows') or 0) > 0]
    empty   = [s for s in tgts if got.get(s) and not (got[s].get('rows') or 0)]
    missing = [s for s in tgts if s not in got]
    L.append(f"| {ep['block']} | {ep['name']} | {len(usable)} | {len(empty)} | "
             f"{len(missing)} | {len(tgts)} |")
    if missing or empty:
        gaps.append((ep['name'], missing, empty))
L.append('')
for name, missing, empty in gaps:
    L.append(f'**{name}**')
    if missing:
        L.append(f'- never fetched ({len(missing)}): {", ".join(missing[:80])}'
                 + (' ...' if len(missing) > 80 else ''))
    if empty:
        L.append(f'- fetched but empty ({len(empty)}): {", ".join(empty[:80])}'
                 + (' ...' if len(empty) > 80 else ''))
    L.append('')

L += ['## Date ranges obtained', '']
for name in ('price_eod_full', 'b1_price_eod_full', 'b1_historical_market_cap',
             'b2_price_eod_full', 'fx_eod'):
    got = by_ep.get(name, {})
    if not got:
        L.append(f'- **{name}**: nothing downloaded.'); continue
    mins = sorted(r['date_min'] for r in got.values() if r.get('date_min'))
    if not mins:
        L.append(f'- **{name}**: no dated rows.'); continue
    years = defaultdict(int)
    for r in got.values():
        if r.get('date_min'):
            years[r['date_min'][:4]] += 1
    top = sorted(years.items(), key=lambda kv: -kv[1])[:3]
    L.append(f'- **{name}**: earliest {mins[0]}; start-year clustering '
             + ', '.join(f'{y} ({n})' for y, n in top)
             + '. One dominant recent year would mean a plan cap, not real history.')
L.append('')

fails = defaultdict(list)
for r in rows:
    if r.get('disposition') == 'FAILED':
        fails[r['endpoint']].append(r.get('http_status'))
L += ['## What the plan refused', '']
L += ['Nothing was refused.'] if not fails else \
     [f'- `{ep}`: {len(v)} failures, e.g. HTTP {v[0]}' for ep, v in sorted(fails.items())]
L.append('')

beside = [r for r in rows if r.get('disposition') == 'WROTE_BESIDE']
def dirsize(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
def human(n):
    for u in ('B','KB','MB','GB'):
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} TB'
L += ['## Size, and the safety guarantees', '',
      f'- raw: `{RAW}` - {human(dirsize(RAW))}',
      f'- sealed DCF: {len(list(VENDOR_DCF.rglob("*.json.gz")))} files, '
      f'{human(dirsize(VENDOR_DCF))}, **not opened**',
      f'- files written beside rather than over an existing file: **{len(beside)}**']
L += [f"  - {r['endpoint']} / {r.get('symbol')} -> {r['file']}" for r in beside] or \
     ['  - none; nothing was overwritten']
L += ['- every symbol list was sorted before iteration; no Python set was iterated',
      '- every response was written to disk exactly as received, before any parsing', '']

cpath = OUT / f'COMPLETENESS_REPORT_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.md'
if cpath.exists():
    cpath = cpath.with_name(cpath.stem + datetime.now(timezone.utc).strftime('__%H%M%SZ') + '.md')
cpath.write_text('\n'.join(L))
print('\n'.join(L))
print('\n\nSaved to:', cpath)

---
## 10. Getting it off Drive, which is the part that matters legally

FMP's confirmation allows retention **on a local computer**. Drive is not that. So:

1. In Drive, right-click `fmp_extraction_2026-09` → **Download**. Google zips it.
2. Unzip it on your Mac, somewhere permanent.
3. Run `code/22_fmp_localise.py` against that folder. It re-hashes every file and compares
   to the manifest, and refuses to pass if a single hash differs.
4. Make the **second local copy** — external drive, or confirm Time Machine has it. Two
   local copies, not one local and one in Drive.
5. Only when the verification passes and the second copy exists: delete the folder from
   Drive, and empty Drive's bin.
6. Only then cancel the subscription — and only after the design chat has read the section
   C numbers.

**Do not use Colab's "Save a copy in GitHub"** at any point. After a run these cells hold
FMP data; saving that back would push licensed data into a public repository. Edit and push
from your machine; use Colab only to run. If you must save from Colab, **Edit → Clear all
outputs** first.